In [1]:
# Load a TIFF image for the first time
# -------------------------------------------------
# This cell demonstrates the most basic operation:
#   * import the required Python packages
#   * specify the file location
#   * read the TIFF into a NumPy array
#   * print a few useful metadata fields
#   * (optionally) display the image if it is 2‑D

# 1️⃣  Imports
import tifffile                     # for reading TIFF files
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt     # for quick visualisation
import numpy as np                  # for array handling
from pathlib import Path            # convenient path manipulation

import napari
import zarr
import dask.array as da
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader

In [2]:
# 1. Configuration 
channels = ["CD45", "CD3E", "CD8a"]
marker_colors = {"CD45": "cyan", "CD3E": "magenta", "CD8a": "lime"}

# 2. Create UI Elements (No observers attached manually)
# We use a dictionary for the checkboxes to map them directly to function arguments
checkbox_dict = {c: widgets.Checkbox(value=False, description=c) for c in channels}
ui_box = widgets.HBox(list(checkbox_dict.values()))

# 3. Define the Plotting Function
# The arguments must match the keys in checkbox_dict
def plot_histogram(CD45, CD3E, CD8a):
    # This is the "Nuclear Option" to ensure only ONE figure exists at a time
    plt.close('all')
    
    # Create a list of active markers based on the checkbox booleans
    active_map = {"CD45": CD45, "CD3E": CD3E, "CD8a": CD8a}
    active_channels = [k for k, v in active_map.items() if v]

    if not active_channels:
        print("Select one or more channels to overlay their distributions.")
        return

    # Initialize the single figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for chan in active_channels:
        path = Path(f'../input/{chan}.tif')
        if not path.exists():
            continue
            
        # Use memmap for memory safety on the MSI EdgeXpert
        img = tifffile.memmap(path, mode='r')
        
        # Sub-sample (::100) for instant UI response
        sample = img[::100, ::100].ravel()
        color = marker_colors.get(chan, "steelblue")
        
        # Overlay on the single axis
        ax.hist(sample, bins=256, range=(0, 65535), color=color, 
                label=chan, alpha=0.5, log=True, edgecolor='none')

    ax.set_title('Overlaid Marker Intensity Distributions (Log Scale)')
    ax.set_xlabel('16-bit Intensity Value')
    ax.set_ylabel('Pixel Count (Log)')
    ax.set_xlim(0, 65535)
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.2)
    plt.show()

# 4. Use interactive_output to link UI and Function
# This manages the threading and prevents the "triple-fire" issue
out = widgets.interactive_output(plot_histogram, checkbox_dict)

# 5. Display everything
display(ui_box, out)

Output()

# 1. Path to your 23-channel Zarr store
zarr_path = "../data/CellDIVE_SLIDE-045.zarr"

def load_spatial_data(url):
    # Parse the OME-Zarr
    parsed_url = parse_url(url, mode="r")
    reader = Reader(parsed_url)
    nodes = list(reader())
    
    # Get the highest resolution (usually the first node)
    image_node = nodes[0]
    data = image_node.data[0] # This is a Dask array (Lazy Loading!)
    
    # Get channel metadata (names/colors)
    metadata = image_node.metadata
    channel_names = metadata.get('name', [f"Ch_{i}" for i in range(data.shape[0])])
    
    return data, channel_names

# 2. Launch Napari with Lazy Layers
data, names = load_spatial_data(zarr_path)
viewer = napari.Viewer()

# Add each channel as a separate layer for easy toggling in the UI
for i, name in enumerate(names):
    viewer.add_image(
        data[i], 
        name=name, 
        blending='additive', 
        visible=False, # Start hidden to save initial rendering time
        contrast_limits=[0, 1000] # Adjust based on your histogram stats
    )

napari.run()